# Entrenar DINOv2 con las vacas de Tres Lomas

Este notebook ajusta **DINOv2 small** (el mismo modelo que usa `campo.py` para la huella) con los recortes que identificaron en la página, y lo compara contra el DINOv2 original.

**Antes de empezar**
1. Subí `dataset_dinov2.zip` a tu Google Drive (suelto en Mi unidad o en la carpeta **vacas**; el notebook lo encuentra solo).
2. Acá arriba: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.
3. **Entorno de ejecución → Ejecutar todas**. La primera vez pide permiso para usar tu Drive: aceptá.

Tarda en total unos 40 a 60 minutos. Todo lo que produce queda en **Mi unidad/vacas/resultado_dinov2/**:
- `dinov2_vacas.pt`: el modelo entrenado (pesos de DINOv2, se carga igual que el original).
- `metricas.json`: el acierto antes y después.
- `enmascarado.zip`: los recortes sin fondo (si volvés a correr, no hace falta recalcularlos).

Si Colab se desconecta a mitad de camino, volvé a hacer *Ejecutar todas*: retoma desde lo que ya quedó guardado en Drive.

In [ ]:
# 1. Drive, GPU y librerías
import os
from google.colab import drive
if os.path.ismount('/content/drive'):
    drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'ultralytics'], check=True)

import torch
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución → Cambiar tipo → T4 GPU, y volvé a ejecutar.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Configuración y descompresión del dataset
import json, zipfile, random, time, csv, shutil, glob
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from PIL import Image, ImageOps, ImageDraw, ImageFilter

MI_UNIDAD = Path('/content/drive/MyDrive')
# Busca dataset_dinov2.zip suelto en Mi unidad o dentro de una carpeta del primer nivel
candidatos = sorted(glob.glob(str(MI_UNIDAD / 'dataset_dinov2.zip')) +
                    glob.glob(str(MI_UNIDAD / '*' / 'dataset_dinov2.zip')))
assert candidatos, ('No encuentro dataset_dinov2.zip en Mi unidad. Fijate que haya terminado de subir '
                    'y volvé a ejecutar la celda 1.')
ZIP = Path(candidatos[0])
if len(candidatos) > 1:
    print('[AVISO] Hay más de un dataset_dinov2.zip; uso', ZIP)
print('Dataset:', ZIP, f'({ZIP.stat().st_size / 1e6:.0f} MB)')

SALIDA = MI_UNIDAD / 'vacas' / 'resultado_dinov2'
SALIDA.mkdir(parents=True, exist_ok=True)
print('Los resultados se guardan en:', SALIDA)
TRABAJO = Path('/content/trabajo')

MODELO_EMB = 'vit_small_patch14_dinov2.lvd142m'
TAM = 224
GRIS = (124, 116, 104)          # mismo fondo neutro que campo.py
MEDIA = np.array([0.485, 0.456, 0.406], np.float32)
DESV = np.array([0.229, 0.224, 0.225], np.float32)
SEMILLA = 7
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

if not (TRABAJO / 'dataset_dinov2' / 'meta.csv').exists():
    print('Descomprimiendo...')
    TRABAJO.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(TRABAJO)
DATA = TRABAJO / 'dataset_dinov2'
meta = list(csv.DictReader(open(DATA / 'meta.csv', encoding='utf-8')))
print(Counter(m['split'] for m in meta))

In [ ]:
# 3. Quitar el fondo con SAM (igual que campo.py) — se guarda en Drive para no repetirlo
from ultralytics import SAM

MASC = TRABAJO / 'enmascarado'
ZIP_MASC = SALIDA / 'enmascarado.zip'
if ZIP_MASC.exists() and not MASC.exists():
    with zipfile.ZipFile(ZIP_MASC) as z:
        z.extractall(TRABAJO)

def caja_con_margen(box, ancho, alto, margen=0.05):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    return (int(max(0, x1 - w * margen)), int(max(0, y1 - h * margen)),
            int(min(ancho, x2 + w * margen)), int(min(alto, y2 + h * margen)))

def recortar_con_mascara(im, box, poligono, margen=0.05):
    # misma función que identificador.py
    c = caja_con_margen(box, im.width, im.height, margen)
    rec = im.crop(c)
    if poligono is None or len(poligono) < 3:
        return rec
    pts = [(float(x) - c[0], float(y) - c[1]) for x, y in poligono]
    mascara = Image.new('L', rec.size, 0)
    ImageDraw.Draw(mascara).polygon(pts, fill=255)
    mascara = mascara.filter(ImageFilter.MaxFilter(5))
    fondo = Image.new('RGB', rec.size, GRIS)
    return Image.composite(rec, fondo, mascara)

def rgba_a_gris(im):
    caja = im.getchannel('A').point(lambda v: 255 if v > 20 else 0).getbbox()
    if caja is None:
        return None
    im = im.crop(caja)
    fondo = Image.new('RGB', im.size, GRIS)
    fondo.paste(im, mask=im.getchannel('A'))
    return fondo

pendientes = [m for m in meta if not (MASC / Path(m['archivo']).with_suffix('.jpg')).exists()]
print(f'Para enmascarar: {len(pendientes)} de {len(meta)}')
if pendientes:
    sam = SAM('sam2.1_t.pt')
    t0, sin_silueta = time.time(), 0
    for i, m in enumerate(pendientes, 1):
        destino = MASC / Path(m['archivo']).with_suffix('.jpg')
        destino.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(DATA / m['archivo']) as im:
            im = ImageOps.exif_transpose(im)
            if m['split'] == 'referencia':
                out = rgba_a_gris(im.convert('RGBA')) if im.mode in ('RGBA', 'LA') else im.convert('RGB')
                out = out or im.convert('RGB')
            else:
                im = im.convert('RGB')
                caja = json.loads(m['caja'])
                poli = None
                try:
                    r = sam.predict(im, bboxes=[caja], device=0, verbose=False)[0]
                    if r.masks is not None and len(r.masks.xy) and len(r.masks.xy[0]) >= 3:
                        poli = r.masks.xy[0].tolist()
                except Exception as e:
                    print('SAM falló en', m['archivo'], e)
                if poli is None:
                    sin_silueta += 1
                out = recortar_con_mascara(im, caja, poli)
        out.save(destino, quality=92)
        if i % 1000 == 0:
            print(f'  {i}/{len(pendientes)}  ({(time.time() - t0) / i * (len(pendientes) - i) / 60:.0f} min restantes)')
    print(f'Listo. Sin silueta (quedan con fondo): {sin_silueta}')
    print('Guardando copia en Drive...')
    with zipfile.ZipFile(ZIP_MASC, 'w', zipfile.ZIP_STORED) as z:
        for p in MASC.rglob('*.jpg'):
            z.write(p, p.relative_to(TRABAJO).as_posix())

for m in meta:
    m['img'] = str(MASC / Path(m['archivo']).with_suffix('.jpg'))

In [ ]:
# 4. Modelo, preparación de imágenes y cálculo de huellas
import timm
import torch.nn as nn
import torch.nn.functional as F
DEV = 'cuda'

def nuevo_modelo():
    return timm.create_model(MODELO_EMB, pretrained=True, num_classes=0, img_size=TAM)

def a_tensor(im):
    # igual que Embebedor._preparar de identificador.py: cuadrado con fondo gris, normalizado
    im = ImageOps.pad(im.convert('RGB'), (TAM, TAM), color=GRIS)
    a = (np.asarray(im, dtype=np.float32) / 255.0 - MEDIA) / DESV
    return torch.from_numpy(a.transpose(2, 0, 1))

@torch.no_grad()
def huellas(modelo, rutas, lote=128):
    # original + espejado, normalizado: lo mismo que hace campo.py
    modelo.eval()
    partes = []
    for i in range(0, len(rutas), lote):
        x = torch.stack([a_tensor(Image.open(r)) for r in rutas[i:i + lote]]).to(DEV)
        with torch.autocast('cuda', dtype=torch.float16):
            f = modelo(x).float() + modelo(torch.flip(x, dims=[3])).float()
        partes.append(F.normalize(f, dim=1).cpu())
    return torch.cat(partes).numpy()

In [ ]:
# 5. Limpieza extra de recorridos largos (pueden haber saltado de una vaca a otra)
# En recorridos de más de 500 frames, se descarta el 20% de recortes que menos se parecen
# al resto del mismo recorrido (con el DINOv2 original).
base = nuevo_modelo().to(DEV)
ent = [m for m in meta if m['split'] == 'entrenar']
largos = defaultdict(list)
for m in ent:
    if int(m['largo_recorrido']) > 500:
        largos[m['recorrido']].append(m)
fuera = set()
for rec, ms in largos.items():
    E = huellas(base, [m['img'] for m in ms])
    c = E.mean(0); c /= np.linalg.norm(c)
    sim = E @ c
    corte = np.quantile(sim, 0.2)
    fuera.update(m['archivo'] for m, s in zip(ms, sim) if s < corte)
print(f'Recorridos largos revisados: {len(largos)}. Recortes descartados: {len(fuera)}')
meta_ok = [m for m in meta if m['archivo'] not in fuera]

In [ ]:
# 6. Prueba: ¿reconoce a la vaca en un recorrido que nunca vio?
# Para cada recorrido de prueba se promedia la huella de sus recortes (como en la página) y se
# compara contra la galería. Puntaje de cada vaca = promedio de sus 3 fotos más parecidas.
# Se mide de dos formas:
#   A) galería = fotos de referencia + recortes del campo de entrenamiento (como la página hoy)
#   B) galería = SOLO fotos de referencia (el caso difícil: una vaca nueva sin nada del campo)

def evaluar(modelo, nombre):
    refs = [m for m in meta_ok if m['split'] == 'referencia']
    entr = [m for m in meta_ok if m['split'] == 'entrenar']
    prue = [m for m in meta_ok if m['split'] == 'prueba']
    E_ref = huellas(modelo, [m['img'] for m in refs])
    E_ent = huellas(modelo, [m['img'] for m in entr])
    E_pru = huellas(modelo, [m['img'] for m in prue])
    por_rec = defaultdict(list)
    for m, e in zip(prue, E_pru):
        por_rec[(m['recorrido'], m['vaca'])].append(e)
    Q = np.stack([np.mean(v, 0) for v in por_rec.values()])
    Q /= np.linalg.norm(Q, axis=1, keepdims=True)
    verdad = [v for (_, v) in por_rec]
    res = {}
    for modo, (G, gv) in {
        'A_referencias_y_campo': (np.concatenate([E_ref, E_ent]), [m['vaca'] for m in refs + entr]),
        'B_solo_referencias': (E_ref, [m['vaca'] for m in refs]),
    }.items():
        gv = np.array(gv)
        vacas = sorted(set(gv))
        S = Q @ G.T
        P = np.stack([np.sort(S[:, gv == v], axis=1)[:, -3:].mean(1) for v in vacas], 1)
        orden = np.argsort(-P, axis=1)
        pos = [list(np.array(vacas)[orden[i]]).index(verdad[i]) for i in range(len(verdad))]
        res[modo] = {'top1': float(np.mean([p == 0 for p in pos])),
                     'top5': float(np.mean([p < 5 for p in pos])),
                     'recorridos': len(pos), 'vacas_candidatas': len(vacas)}
    print(f'--- {nombre}')
    for modo, r in res.items():
        print(f"  {modo}: acierto 1°: {r['top1']:.0%}   en top 5: {r['top5']:.0%}   "
              f"({r['recorridos']} recorridos, {r['vacas_candidatas']} vacas candidatas)")
    return res

metricas = {'antes': evaluar(base, 'DINOv2 ORIGINAL')}

In [ ]:
# 7. Entrenamiento
# - Se entrenan solo los últimos 4 bloques de DINOv2 (el resto queda como viene): aprende lo propio
#   de las vacas sin olvidar lo general.
# - Pérdida ArcFace: junta las fotos de la misma vaca y separa las de vacas distintas.
# - Aumentos: rotación libre (desde arriba la vaca puede estar en cualquier dirección), espejado,
#   cambios de luz y, a veces, blanco y negro (para que también sirva de noche con infrarrojo).
import torchvision.transforms as T

EPOCAS = 12
LOTE = 64
BLOQUES_ENTRENABLES = 4

train = [m for m in meta_ok if m['split'] in ('entrenar', 'referencia')]
vacas = sorted({m['vaca'] for m in train})
idx = {v: i for i, v in enumerate(vacas)}
print(f'Entrenamiento: {len(train)} imágenes de {len(vacas)} vacas')

color = T.Compose([T.RandomApply([T.ColorJitter(0.35, 0.35, 0.25, 0.03)], p=0.8),
                   T.RandomGrayscale(p=0.2),
                   T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2)])

class Datos(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        m = self.items[i]
        im = Image.open(m['img']).convert('RGB')
        ang = random.uniform(-180, 180) if m['split'] != 'referencia' else random.uniform(-15, 15)
        im = im.rotate(ang, resample=Image.BILINEAR, expand=True, fillcolor=GRIS)
        w, h = im.size
        s = random.uniform(0.8, 1.0)
        cw, ch = int(w * s), int(h * s)
        x0, y0 = random.randint(0, w - cw), random.randint(0, h - ch)
        im = im.crop((x0, y0, x0 + cw, y0 + ch))
        if random.random() < 0.5:
            im = ImageOps.mirror(im)
        im = color(im)
        return a_tensor(im), idx[m['vaca']]

# muestreo: cada vaca sale con probabilidad proporcional a la raíz de su cantidad de fotos
cuenta = Counter(m['vaca'] for m in train)
pesos = [1.0 / np.sqrt(cuenta[m['vaca']]) for m in train]
muestreador = torch.utils.data.WeightedRandomSampler(pesos, num_samples=len(train), replacement=True)
cargador = torch.utils.data.DataLoader(Datos(train), batch_size=LOTE, sampler=muestreador,
                                       num_workers=2, drop_last=True, persistent_workers=True)

class ArcFace(nn.Module):
    def __init__(self, dim, n, s=30.0, m=0.30):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n, dim) * 0.01)
        self.s, self.m = s, m
    def forward(self, f, y):
        cos = F.linear(F.normalize(f), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        tita = torch.acos(cos)
        objetivo = torch.cos(tita + self.m)
        uno = F.one_hot(y, cos.shape[1]).bool()
        return F.cross_entropy(self.s * torch.where(uno, objetivo, cos), y)

modelo = nuevo_modelo().to(DEV)
for p in modelo.parameters():
    p.requires_grad = False
entrenables = list(modelo.blocks[-BLOQUES_ENTRENABLES:]) + [modelo.norm]
for mod in entrenables:
    for p in mod.parameters():
        p.requires_grad = True
cabeza = ArcFace(modelo.num_features, len(vacas)).to(DEV)

opt = torch.optim.AdamW([
    {'params': [p for mod in entrenables for p in mod.parameters()], 'lr': 2e-5},
    {'params': cabeza.parameters(), 'lr': 1e-3}], weight_decay=0.05)
pasos = EPOCAS * len(cargador)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[2e-5, 1e-3], total_steps=pasos, pct_start=0.1)
escalador = torch.cuda.amp.GradScaler()

CHECK = SALIDA / 'checkpoint.pt'
inicio = 0
if CHECK.exists():
    ck = torch.load(CHECK, map_location=DEV)
    if ck.get('vacas') == vacas:
        modelo.load_state_dict(ck['modelo']); cabeza.load_state_dict(ck['cabeza'])
        opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        inicio = ck['epoca'] + 1
        print(f'Retomando desde la época {inicio + 1}')

for ep in range(inicio, EPOCAS):
    modelo.train(); t0 = time.time(); total = 0.0
    for x, y in cargador:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            f = modelo(x)
        perdida = cabeza(f.float(), y)
        opt.zero_grad(set_to_none=True)
        escalador.scale(perdida).backward()
        escalador.step(opt); escalador.update(); sched.step()
        total += perdida.item()
    print(f'Época {ep + 1}/{EPOCAS}  pérdida {total / len(cargador):.3f}  ({(time.time() - t0) / 60:.1f} min)')
    torch.save({'modelo': modelo.state_dict(), 'cabeza': cabeza.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'epoca': ep, 'vacas': vacas}, CHECK)

In [ ]:
# 8. Comparar y guardar
metricas['despues'] = evaluar(modelo, 'DINOv2 ENTRENADO')
metricas['config'] = {'modelo_base': MODELO_EMB, 'epocas': EPOCAS, 'bloques_entrenables': BLOQUES_ENTRENABLES,
                      'imagenes_entrenamiento': len(train), 'vacas': len(vacas),
                      'descartados_recorridos_largos': len(fuera)}

print('\nResumen (acierto en 1° lugar):')
for modo in metricas['antes']:
    a, d = metricas['antes'][modo]['top1'], metricas['despues'][modo]['top1']
    print(f'  {modo}: {a:.0%} → {d:.0%}')

# Pesos de DINOv2 solamente: se cargan con
#   timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=False, num_classes=0, img_size=224)
#   modelo.load_state_dict(torch.load('dinov2_vacas.pt'))
torch.save({k: v.half() for k, v in modelo.state_dict().items()}, SALIDA / 'dinov2_vacas.pt')
json.dump(metricas, open(SALIDA / 'metricas.json', 'w'), indent=1, ensure_ascii=False)
print(f'\nGuardado en {SALIDA}: dinov2_vacas.pt y metricas.json')